# Prerequisites: Data Preparation & Baseline Evaluation (~15 min on 2x H200)

This notebook has two goals:
1. **Prepare the distillation dataset** — download [WikiText-103](https://huggingface.co/datasets/Salesforce/wikitext/tree/main/wikitext-103-v1) and tokenize it into the binary format expected by Megatron-Bridge. This dataset is used during the distillation step (after pruning) in all scenario notebooks.
2. **Establish the teacher baseline** — evaluate the original Qwen3-8B on MMLU before any compression.

> **Why prepare the dataset before compression?** The distillation step (which comes *after* pruning) requires a pre-tokenized dataset in Megatron binary format. Preparing it upfront avoids interrupting the compression pipeline and ensures a consistent dataset across all scenarios.

> **Note on calibration datasets:** Pruning also requires calibration data to score the importance of each component — the model runs forward passes on a small dataset to measure how much each neuron, head, or layer contributes to the output. This calibration data (we use [`nvidia/Nemotron-Post-Training-Dataset-v2`](https://huggingface.co/datasets/nvidia/Nemotron-Post-Training-Dataset-v2)) is handled separately in each scenario notebook. Minitron downloads it automatically under the hood, while Puzzletron requires an explicit preparation step. See the respective notebooks ([`scenario1_minitron.ipynb`](scenario1_minitron.ipynb), [`scenario2_puzzletron.ipynb`](scenario2_puzzletron.ipynb), etc.) for details.

**Prerequisites:** Run this notebook from inside the NeMo container with the base model already downloaded at `/workspace/models/Qwen3-8B` (see the guide's Prerequisites section).

## 1. Download and tokenize distillation dataset

For distillation we use [WikiText-103](https://huggingface.co/datasets/Salesforce/wikitext/tree/main/wikitext-103-v1) — a small, generic language modeling dataset.

The `megatron_preprocess_data` utility downloads the dataset directly from the HuggingFace Hub and tokenizes it into the binary `.bin` / `.idx` format expected by Megatron-Bridge in a single step (~2 min).

In [ ]:
!python -m modelopt.torch.utils.plugins.megatron_preprocess_data \
    --hf_dataset wikitext \
    --hf_name wikitext-103-v1 \
    --hf_split train \
    --json_keys text \
    --tokenizer /workspace/models/Qwen3-8B \
    --output_dir /workspace/datasets/tokenized_qwen3 \
    --workers 32 \
    --append_eod \
    --strip_newlines

## 2. Evaluate teacher model (baseline)

Before compressing, we establish the baseline MMLU score for the original Qwen3-8B. Results in the guide are expressed as a percentage of this number.

We use [`lm-eval`](https://github.com/EleutherAI/lm-evaluation-harness) — a standard open-source evaluation harness — to run the MMLU benchmark. MMLU (Massive Multitask Language Understanding) covers 57 subjects across STEM, humanities, and social sciences, using 4-choice questions. The 5-shot setting provides 5 in-context examples per question, which is the standard configuration for comparing LLMs on this benchmark.

In [ ]:
!lm_eval --model hf \
    --model_args pretrained=/workspace/models/Qwen3-8B,dtype=bfloat16 \
    --tasks mmlu \
    --num_fewshot 5 \
    --batch_size 4

**Expected result:** MMLU (5-shot) = **0.7493**. This is the teacher baseline used throughout the guide.